# 📊 สมุดบันทึกที่ 1: การสำรวจและทำความสะอาดข้อมูล (EDA & Data Cleaning)
**วิชา:** การทำเหมืองข้อมูล (Data Mining)  
**เนื้อหาอ้างอิง:** บทที่ 1 (KDD Process), บทที่ 2 (การเตรียมข้อมูล), และบทที่ 3 (EDA & Data Visualization)  
**เป้าหมาย:** สำรวจโครงสร้าง ตรวจสอบคุณภาพข้อมูล (Missing, Duplicates, Outliers), วิเคราะห์สถิติ และเตรียมคุณลักษณะก่อนสร้างโมเดลจำแนกผลการเรียน


## 1. การนำเข้าไลบรารีที่จำเป็น (Import Libraries)
อ้างอิงตามบทที่ 2 และบทที่ 3: ใช้ `pandas` จัดการข้อมูล, `numpy` คำนวณทางสถิติ, `matplotlib` และ `seaborn` สำหรับพล็อตแสดงผล


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# กำหนดรูปแบบการแสดงผลของกราฟ
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
print("✅ นำเข้าไลบรารีเรียบร้อย")


## 2. ขั้นตอนที่ 1: การตรวจสอบโครงสร้างข้อมูล (Data Structuring - บทที่ 3.2.1)
โหลดข้อมูลจาก `student-por.csv` (วิชาภาษาโปรตุเกส) และตรวจสอบขนาด แถว คอลัมน์ และตัวอย่างข้อมูลเบื้องต้น


In [ ]:
# 1. โหลดข้อมูล (รองรับทั้ง delimiter แบบ ';' และ ',')
try:
    df = pd.read_csv('../data/student-por.csv', sep=';')
    if df.shape[1] == 1:
        df = pd.read_csv('../data/student-por.csv', sep=',')
except:
    df = pd.read_csv('../data/student-por.csv')

print(f"📊 ขนาดข้อมูล: มีทั้งหมด {df.shape[0]} แถว (นักเรียน) และ {df.shape[1]} คอลัมน์")
df.head()


### 2.1 ตรวจสอบประเภทตัวแปร (Stevens' Typology - บทที่ 2.4.1)
ตรวจสอบ Data Type เพื่อจำแนกว่าตัวแปรใดเป็น Nominal, Binary, Ordinal หรือ Ratio


In [ ]:
# ตรวจสอบ Data Type และ Non-Null Count
print("=" * 60)
print("📋 โครงสร้างและชนิดของตัวแปร (Data Types):")
print("=" * 60)
df.info()


In [ ]:
# แสดงสถิติเชิงพรรณนาสำหรับตัวแปรเชิงปริมาณ
df.describe()


## 3. ขั้นตอนที่ 2: การทำความสะอาดข้อมูล (Data Cleaning - บทที่ 2.6 & 3.2 ขั้นที่ 2)

### 3.1 ตรวจสอบข้อมูลสูญหาย (Missing Values - บทที่ 2.6.2)
ตรวจสอบว่ามีข้อมูลสูญหายหรือไม่ และเข้าข่ายประเภทใด (MCAR, MAR, MNAR)


In [ ]:
# ตรวจสอบจำนวน Missing Values ในแต่ละคอลัมน์
missing_count = df.isnull().sum()
total_missing = missing_count.sum()

print(f"🔍 จำนวน Missing Values ทั้งหมดในชุดข้อมูล: {total_missing} ค่า")
if total_missing == 0:
    print("✨ ข้อมูลมีความสมบูรณ์แบบ 100% ไม่จำเป็นต้องทำ Data Imputation")
else:
    print(missing_count[missing_count > 0])


### 3.2 ตรวจสอบข้อมูลซ้ำซ้อน (Duplicate Data - บทที่ 2.6.3)


In [ ]:
# ตรวจสอบแถวที่ซ้ำกัน
duplicate_count = df.duplicated().sum()
print(f"🔍 จำนวนแถวที่ซ้ำซ้อน: {duplicate_count} แถว")
if duplicate_count == 0:
    print("✨ ไม่พบข้อมูลซ้ำซ้อนในชุดข้อมูล")


### 3.3 การตรวจจับและจัดการค่าผิดปกติ (Outlier Detection ด้วยวิธี IQR - บทที่ 2.6.1)
ตามบทที่ 2.6.1 กำหนดให้ใช้สูตร Interquartile Range (IQR):
$$IQR = Q_3 - Q_1$$
$$	ext{Lower Bound} = Q_1 - 1.5 	imes IQR \quad , \quad 	ext{Upper Bound} = Q_3 + 1.5 	imes IQR$$


In [ ]:
# ฟังก์ชันตรวจจับ Outlier ด้วย IQR Method ตามบทที่ 2.6.1
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# ตรวจสอบตัวแปรเชิงตัวเลขสำคัญ
numeric_features = ['age', 'absences', 'failures', 'G3']
print("📊 ผลการตรวจสอบ Outliers ด้วยวิธี IQR:")
print("-" * 65)
for col in numeric_features:
    count, low, up = detect_outliers_iqr(df, col)
    print(f"• {col:<10}: พบ {count:>3} ค่า | ช่วงปกติ [{low:>5.1f}, {up:>5.1f}] | ต่ำสุด-สูงสุด [{df[col].min()}, {df[col].max()}]")


#### 💡 การตัดสินใจเชิงวิเคราะห์เรื่อง Outlier (สำคัญมาก!):
* ในตัวแปร `absences` พบ Outlier 21 คน (ขาดเรียนเกิน 15 วันขึ้นไป จนถึง 32 วัน)
* ในตัวแปร `failures` พบ Outlier 100 คน (เคยสอบตก $\ge 1$ ครั้ง)
* **การตัดสินใจ:** ตามบทที่ 2.6.1 กรณีที่ Outlier ไม่ใช่ข้อมูลบันทึกผิด (Noise) แต่เป็น **พฤติกรรมจริงของกลุ่มเสี่ยงที่เราต้องการทำนาย (Valid Outlier)** เรา **ต้องเก็บไว้ทั้งหมด ห้ามลบทิ้ง** เพราะหากลบทิ้ง โมเดลจะไม่เคยเห็นพฤติกรรมเด็กที่ขาดเรียนหนักๆ หรือเคยสอบตกเลย


## 4. ขั้นตอนที่ 3: การสำรวจข้อมูลเชิงตัวแปรเดี่ยว (Univariate Analysis - บทที่ 3.2 ขั้นที่ 3)

### 4.1 การสร้างตัวแปรเป้าหมาย (Target Binarization - บทที่ 2.7.7)
แปลงเกรด $G3$ ให้เป็น Binary Class ตามเกณฑ์ผ่านของโปรตุเกส:
* $	ext{Pass (1): } G3 \ge 10$
* $	ext{Fail (0): } G3 < 10$


In [ ]:
# สร้างคอลัมน์ passed
df['passed'] = (df['G3'] >= 10).astype(int)

# สรุปจำนวนและสัดส่วน
pass_counts = df['passed'].value_counts()
pass_props = df['passed'].value_counts(normalize=True) * 100

print(f"🟢 สอบผ่าน (Pass - 1)  : {pass_counts[1]} คน ({pass_props[1]:.2f}%)")
print(f"🔴 สอบไม่ผ่าน (Fail - 0): {pass_counts[0]} คน ({pass_props[0]:.2f}%)")

# พล็อตดูกราฟสัดส่วน Imbalanced Data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='passed', data=df, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Target Distribution: Pass (1) vs Fail (0)')
axes[0].set_xticklabels(['Fail (<10)', 'Pass (>=10)'])

sns.histplot(df['G3'], bins=20, kde=True, color='#3498db', ax=axes[1])
axes[1].axvline(10, color='red', linestyle='--', label='Pass Threshold (10)')
axes[1].set_title('Distribution of Final Grade (G3)')
axes[1].legend()
plt.tight_layout()
plt.show()


## 5. ขั้นตอนที่ 4: การวิเคราะห์ความสัมพันธ์สองตัวแปร (Bivariate Analysis - บทที่ 3.2 ขั้นที่ 4)


In [ ]:
# 5.1 วิเคราะห์ Correlation Matrix
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
sns.heatmap(corr[['G1', 'G2', 'G3', 'passed']].sort_values(by='G3', ascending=False), 
            annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation with Target Grades (G1, G2, G3, passed)')
plt.show()


#### ⚠️ ข้อสรุปเรื่อง Data Leakage (ตามโจทย์ข้อ 4):
สังเกตว่า $G1$ มี Correlation กับ $G3$ สูงถึง 0.83 และ $G2$ สูงถึง 0.92  
**ข้อตกลง:** เราจะตัด $G1, G2, G3$ ทิ้ง เพื่อบังคับให้โมเดลทำนายจากพฤติกรรม สภาพครอบครัว และการขาดเรียนล้วนๆ เป็นระบบเตือนภัยล่วงหน้า (Early Warning System) ตั้งแต่วันเปิดเทอม


In [ ]:
# 5.2 พล็อต Boxplot เปรียบเทียบพฤติกรรมสำคัญกับเกรด G3
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(x='failures', y='G3', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Past Failures vs Final Grade')

sns.boxplot(x='higher', y='G3', data=df, palette='Set1', ax=axes[1])
axes[1].set_title('Want Higher Education vs Final Grade')

sns.boxplot(x='studytime', y='G3', data=df, palette='Pastel1', ax=axes[2])
axes[2].set_title('Study Time vs Final Grade')

plt.tight_layout()
plt.show()


## 6. ขั้นตอนที่ 5: การเตรียมคุณลักษณะและการแบ่งชุดข้อมูล (Feature Engineering & Splitting - บทที่ 2.7)


In [ ]:
# 1. แยก Features (X) และ Target (y) พร้อมตัด G1, G2, G3
X_raw = df.drop(columns=['G1', 'G2', 'G3', 'passed'])
y = df['passed']

# 2. ทำ One-Hot Encoding สำหรับตัวแปรประเภท Categorical
X = pd.get_dummies(X_raw, drop_first=True)
print(f"⚙️ จำนวน Features หลังทำ One-Hot Encoding: {X.shape[1]} ตัวแปร")

# 3. สุ่มแบ่งชุดข้อมูล Train/Test (80:20) ด้วย Stratified Sampling (บทที่ 2.7.3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📦 ชุดฝึกสอน (Train Set): {X_train.shape[0]} ตัวอย่าง (ผ่าน {sum(y_train==1)}, ตก {sum(y_train==0)})")
print(f"📦 ชุดทดสอบ (Test Set) : {X_test.shape[0]} ตัวอย่าง (ผ่าน {sum(y_test==1)}, ตก {sum(y_test==0)})")

# 4. บันทึกชุดข้อมูลพร้อมใช้ไปยังโฟลเดอร์ data/processed/
train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/processed/train.csv', index=False)
test_data.to_csv('../data/processed/test.csv', index=False)
print("💾 บันทึก train.csv และ test.csv สำเร็จ พร้อมสำหรับสร้างโมเดลในสมุดบันทึกถัดไป!")
